# Radar Pose Inference
This notebook loads a trained checkpoint from the local `checkpoint/` folder and visualises predictions on individual radar frames.

In [1]:
import os
import json
import math
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets
from tqdm.auto import tqdm


In [2]:
# Locate data and checkpoint folders relative to this notebook
NOTEBOOK_DIR = Path.cwd()


CANDIDATE_CKPT_SUBDIRS = ("checkpoint", "checkpoints")
candidate_ckpt_dirs = []
for parent in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    for sub in CANDIDATE_CKPT_SUBDIRS:
        candidate_ckpt_dirs.append(parent / sub)
CKPT_DIR = next((p for p in candidate_ckpt_dirs if p.exists()), None)
if CKPT_DIR is None:
    raise FileNotFoundError("Could not locate a checkpoint directory. Expected one of: {}".format(
        ", ".join(str(p) for p in candidate_ckpt_dirs)))

# Optional separate checkpoint directory for ResNet-based model
CANDIDATE_RESNET_CKPT_SUBDIRS = ("checkpoint_resnet",)
candidate_resnet_ckpt_dirs = []
for parent in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    for sub in CANDIDATE_RESNET_CKPT_SUBDIRS:
        candidate_resnet_ckpt_dirs.append(parent / sub)
CKPT_DIR_RESNET = next((p for p in candidate_resnet_ckpt_dirs if p.exists()), None)

config_path = CKPT_DIR / "run_config_last.json"
if config_path.exists():
    with config_path.open("r", encoding="utf-8") as fp:
        CFG = json.load(fp)
else:
    CFG = {}

candidate_data_dirs = []
env_dir = os.environ.get("RADAR_DATA_DIR")
if env_dir:
    candidate_data_dirs.append(Path(env_dir))
candidate_data_dirs.append(NOTEBOOK_DIR / "P1")
for parent in NOTEBOOK_DIR.parents:
    candidate_data_dirs.append(parent / "P1")
DATA_DIR = next((p for p in candidate_data_dirs if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate radar data. Configure RADAR_DATA_DIR or place a 'P1' folder near the notebook.")

radar_files = sorted(DATA_DIR.rglob("*_radar.npz"))
FRAME_IDS = [f.relative_to(DATA_DIR).as_posix().removesuffix("_radar.npz") for f in radar_files]
print(f"Found {len(FRAME_IDS)} radar frames under {DATA_DIR}")
preview = FRAME_IDS[:5]
if preview:
    print("Example frame ids:")
    for fid in preview:
        print(" -", fid)
else:
    print("No frames detected. Populate the data folder before running inference.")

# Index of the frame to visualise - change this and re-run the cell below.
FRAME_INDEX = 0

SPLIT_CACHE_PATH = CKPT_DIR / "split_ids.json"


def _load_cached_split(path: Path):
    if path.exists():
        with path.open("r", encoding="utf-8") as fp:
            cached = json.load(fp)
        if all(key in cached for key in ("train", "val", "test")):
            return cached
        print(f"Cached split file at {path} is missing expected keys; ignoring.")
    return None


def _build_training_split(frame_ids):
    shuffled = list(frame_ids)
    rng = random.Random(42)
    rng.shuffle(shuffled)

    n_total = len(shuffled)
    split_train = int(0.8 * n_total)
    split_val = int(0.9 * n_total)

    ids_tr = shuffled[:split_train]
    ids_va = shuffled[split_train:split_val]
    ids_te = shuffled[split_val:]

    if len(ids_te) == 0 and len(ids_va) > 0:
        ids_te = ids_va[-1:]
        ids_va = ids_va[:-1]
    elif len(ids_te) == 0 and len(ids_tr) > 1:
        ids_te = ids_tr[-1:]
        ids_tr = ids_tr[:-1]

    return {"train": ids_tr, "val": ids_va, "test": ids_te}


split_ids = _load_cached_split(SPLIT_CACHE_PATH)
if split_ids is None:
    split_ids = _build_training_split(FRAME_IDS)
    with SPLIT_CACHE_PATH.open("w", encoding="utf-8") as fp:
        json.dump(split_ids, fp, indent=2)
    print(f"Cached split listing to {SPLIT_CACHE_PATH}")
else:
    print(f"Loaded split listing from {SPLIT_CACHE_PATH}")

TRAIN_FRAME_IDS = split_ids["train"]
VAL_FRAME_IDS = split_ids["val"]
TEST_FRAME_IDS = split_ids["test"]

all_split_ids = set(TRAIN_FRAME_IDS) | set(VAL_FRAME_IDS) | set(TEST_FRAME_IDS)
current_ids = set(FRAME_IDS)
extra_cached = all_split_ids - current_ids
missing_cached = current_ids - all_split_ids
if extra_cached:
    print(f"Warning: {len(extra_cached)} cached frames missing from current data directory.")
if missing_cached:
    print(f"Warning: {len(missing_cached)} frames found on disk were not present in the cached split.")

print(f"Train split frames: {len(TRAIN_FRAME_IDS)}")
print(f"Validation split frames: {len(VAL_FRAME_IDS)}")
print(f"Test split frames: {len(TEST_FRAME_IDS)}")


Found 107902 radar frames under m:\MIAMI\projects\codex_new\internship projects\Pose Detucting AI using Radardata\P1
Example frame ids:
 - d1s1/000/00000
 - d1s1/000/00001
 - d1s1/000/00002
 - d1s1/000/00003
 - d1s1/000/00004
Loaded split listing from m:\MIAMI\projects\codex_new\internship projects\Pose Detucting AI using Radardata\checkpoint\split_ids.json
Train split frames: 86321
Validation split frames: 10790
Test split frames: 10791


In [3]:
CAMERA_WIDTH = CFG.get("camera_width", 640)
CAMERA_HEIGHT = CFG.get("camera_height", 480)
LOG_SCALE_INPUT = bool(CFG.get("log_scale_input", True))
CONF_THRESHOLD = float(CFG.get("conf_threshold", 0.5))
PCK_ALPHA = float(CFG.get("metrics_pck_alpha", 0.05))


SKELETON_EDGES = [
    (15, 13), (13, 11), (16, 14), (14, 12),
    (11, 12), (5, 11), (6, 12), (5, 6),
    (5, 7), (7, 9), (6, 8), (8, 10),
    (1, 2), (0, 1), (0, 2), (1, 3), (2, 4),
]


def load_radar_frame(root: Path, fid: str, out_res: int, log_scale: bool = True) -> np.ndarray:
    npz_path = root / f"{fid}_radar.npz"
    with np.load(npz_path) as data:
        hm_hori = data["hm_hori"].astype(np.float32)
        hm_vert = data["hm_vert"].astype(np.float32)
    res_hori = cv2.resize(hm_hori, (out_res, out_res), interpolation=cv2.INTER_LINEAR)
    res_vert = cv2.resize(hm_vert, (out_res, out_res), interpolation=cv2.INTER_LINEAR)
    stacked = np.stack([res_hori, res_vert], axis=0)
    if log_scale:
        stacked = np.log1p(np.clip(stacked, a_min=0.0, a_max=None))
    return stacked


def load_pose_sample(root: Path, fid: str):
    npz_path = root / f"{fid}_pose.npz"
    if not npz_path.exists():
        return None, None
    with np.load(npz_path) as pose_file:
        pose = pose_file["kp"]
    kp = pose[0] if pose.ndim == 3 else pose.astype(np.float32)
    xy = kp[:, :2].astype(np.float32)
    vis = (kp[:, 2] > 0).astype(np.float32)
    return xy, vis


def normalise_channels(inp: np.ndarray, mean: np.ndarray | None, std: np.ndarray | None) -> np.ndarray:
    if mean is not None and std is not None:
        return (inp - mean[:, None, None]) / (std[:, None, None] + 1e-6)
    ch_mean = inp.mean(axis=(1, 2), keepdims=True)
    ch_std = inp.std(axis=(1, 2), keepdims=True) + 1e-6
    return (inp - ch_mean) / ch_std


def load_mask_frame(root: Path, fid: str, out_res: int) -> np.ndarray | None:
    npz_path = root / f"{fid}_mask.npz"
    if not npz_path.exists():
        return None
    with np.load(npz_path) as data:
        # Prefer an explicit 'mask' key, otherwise fall back to first array
        if "mask" in data.files:
            mask = data["mask"].astype(np.float32)
        else:
            first_key = data.files[0]
            mask = data[first_key].astype(np.float32)
    if mask.ndim == 3:
        mask = mask[0]
    mask_resized = cv2.resize(mask, (out_res, out_res), interpolation=cv2.INTER_NEAREST)
    return mask_resized


In [4]:
class SoftArgmax2D(nn.Module):
    def __init__(self, beta: float = 1.0) -> None:
        super().__init__()
        self.beta = beta

    def forward(self, hms: torch.Tensor):
        b, k, h, w = hms.shape
        flat = torch.softmax(hms.view(b * k, -1) * self.beta, dim=-1)
        flat = flat.view(b * k, 1, h, w)
        xs = torch.linspace(0, w - 1, w, device=hms.device).view(1, 1, 1, w)
        ys = torch.linspace(0, h - 1, h, device=hms.device).view(1, 1, h, 1)
        coord_x = torch.sum(flat * xs, dim=[2, 3]).view(b, k, 1)
        coord_y = torch.sum(flat * ys, dim=[2, 3]).view(b, k, 1)
        coords = torch.cat([coord_x, coord_y], dim=-1)
        conf, _ = torch.max(flat.view(b * k, -1), dim=-1)
        return coords, conf.view(b, k)


def conv_bn_relu(cin: int, cout: int, k: int = 3, s: int = 1, p: int = 1) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, s, p, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class ResidualBlock(nn.Module):
    def __init__(self, channels: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.act = nn.ReLU(inplace=True)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.drop(out)
        out = out + identity
        return self.act(out)


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 8) -> None:
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = self.avg_pool(x)
        weights = self.mlp(weights)
        return x * weights


class RadarPoseNet(nn.Module):
    def __init__(self, num_kp: int = 17, in_ch: int = 2, base: int = 64, out_res: int = 128, dropout: float = 0.1) -> None:
        super().__init__()
        self.out_res = out_res
        bottleneck_drop = dropout * 1.5

        self.stem = nn.Sequential(
            conv_bn_relu(in_ch, base, 7, 2, 3),
            ResidualBlock(base, dropout=dropout),
            conv_bn_relu(base, base, 3, 1, 1),
        )
        self.down1 = nn.Sequential(
            conv_bn_relu(base, base * 2, 3, 2, 1),
            ResidualBlock(base * 2, dropout=dropout),
        )
        self.down2 = nn.Sequential(
            conv_bn_relu(base * 2, base * 4, 3, 2, 1),
            ResidualBlock(base * 4, dropout=bottleneck_drop),
            conv_bn_relu(base * 4, base * 4, 3, 1, 1),
            ResidualBlock(base * 4, dropout=bottleneck_drop),
        )
        self.attn = ChannelAttention(base * 4)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(base * 4, base * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.ReLU(True),
            ResidualBlock(base * 2, dropout=dropout),
            nn.ConvTranspose2d(base * 2, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(True),
            ResidualBlock(base, dropout=dropout),
            nn.ConvTranspose2d(base, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(True),
        )
        self.dropout_head = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.hm_head = nn.Conv2d(base, num_kp, 1)
        self.soft = SoftArgmax2D(beta=10.0)

    def forward(self, x: torch.Tensor):
        feats = self.stem(x)
        feats = self.down1(feats)
        feats = self.down2(feats)
        feats = self.attn(feats)
        heat = self.deconv(feats)
        heat = self.dropout_head(heat)
        hm = self.hm_head(heat)
        coords, conf = self.soft(hm)
        return hm, coords, conf


class LegacyRadarPoseNet(nn.Module):
    def __init__(self, num_kp: int = 17, in_ch: int = 2, base: int = 64, out_res: int = 128):
        super().__init__()
        self.out_res = out_res
        self.backbone = nn.Sequential(
            conv_bn_relu(in_ch, base, 7, 2, 3),
            conv_bn_relu(base, base, 3, 1, 1),
            conv_bn_relu(base, base * 2, 3, 2, 1),
            conv_bn_relu(base * 2, base * 2, 3, 1, 1),
            conv_bn_relu(base * 2, base * 4, 3, 2, 1),
            conv_bn_relu(base * 4, base * 4, 3, 1, 1),
            conv_bn_relu(base * 4, base * 4, 3, 1, 1),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(base * 4, base * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base * 2, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(True),
            nn.ConvTranspose2d(base, base, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(True),
        )
        self.hm_head = nn.Conv2d(base, num_kp, 1)
        self.soft = SoftArgmax2D(beta=10.0)

    def forward(self, x: torch.Tensor):
        f = self.backbone(x)
        h = self.deconv(f)
        hm = self.hm_head(h)
        coords, conf = self.soft(hm)
        return hm, coords, conf


def _get_resnet(arch: str = "resnet18", pretrained: bool = False):
    arch = arch.lower()
    try:
        if arch == "resnet18":
            base = torchvision.models.resnet18(weights=(
                torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
            ))
        elif arch == "resnet34":
            base = torchvision.models.resnet34(weights=(
                torchvision.models.ResNet34_Weights.DEFAULT if pretrained else None
            ))
        elif arch == "resnet50":
            base = torchvision.models.resnet50(weights=(
                torchvision.models.ResNet50_Weights.DEFAULT if pretrained else None
            ))
        else:
            raise ValueError(f"Unsupported arch: {arch}")
    except Exception:
        ctor = getattr(torchvision.models, arch)
        base = ctor(pretrained=pretrained)
    return base


class ResNetPoseNet(nn.Module):
    def __init__(
        self,
        num_kp: int = 17,
        in_ch: int = 2,
        out_res: int = 128,
        arch: str = "resnet18",
        pretrained: bool = False,
        train_backbone: bool = True,
    ) -> None:
        super().__init__()
        base = _get_resnet(arch, pretrained=pretrained)
        if in_ch != 3:
            old_conv = base.conv1
            new_conv = nn.Conv2d(
                in_ch,
                old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=(old_conv.bias is not None),
            )
            with torch.no_grad():
                if old_conv.weight.shape[1] == 3:
                    w = old_conv.weight.data
                    w_mean = w.mean(dim=1, keepdim=True)
                    new_w = w_mean.repeat(1, in_ch, 1, 1)
                    new_conv.weight.copy_(new_w)
                else:
                    nn.init.kaiming_normal_(
                        new_conv.weight, mode="fan_out", nonlinearity="relu"
                    )
                if new_conv.bias is not None and old_conv.bias is not None:
                    new_conv.bias.copy_(old_conv.bias.data)
            base.conv1 = new_conv

        feat_channels = base.fc.in_features
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        if not train_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(feat_channels, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
        )
        self.hm_head = nn.Conv2d(64, num_kp, 1)
        self.soft = SoftArgmax2D(beta=10.0)
        self.out_res = out_res

    def forward(self, x: torch.Tensor):
        f = self.backbone(x)
        h = self.deconv(f)
        hm = self.hm_head(h)
        coords, conf = self.soft(hm)
        return hm, coords, conf


In [5]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- load baseline CNN checkpoint ---
ckpt_path = CKPT_DIR / "radar_pose_best.pt"
if not ckpt_path.exists():
    raise FileNotFoundError(f"Expected checkpoint at {ckpt_path}. Run training first or copy the file here.")

ckpt = torch.load(ckpt_path, map_location=device)
state_dict = ckpt["model"]

out_res = int(ckpt.get("out_res", CFG.get("out_res", 128)))
base_channels = CFG.get("base_channels", 64)
dropout = CFG.get("model_dropout", 0.0)
num_kp = CFG.get("num_kp", 17)

uses_legacy = any(key.startswith("backbone") for key in state_dict.keys())
if uses_legacy:
    print("Detected legacy checkpoint; instantiating LegacyRadarPoseNet")
    model = LegacyRadarPoseNet(num_kp=num_kp, in_ch=2, base=base_channels, out_res=out_res)
else:
    print("Detected enhanced checkpoint; instantiating RadarPoseNet")
    model = RadarPoseNet(num_kp=num_kp, in_ch=2, base=base_channels, out_res=out_res, dropout=dropout)

try:
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
except RuntimeError as err:
    raise RuntimeError("Failed to load checkpoint weights into the selected architecture") from err

if missing or unexpected:
    print("State dict discrepancies detected during load:")
    if missing:
        print(" - Missing keys:", missing)
    if unexpected:
        print(" - Unexpected keys:", unexpected)
else:
    print("Checkpoint weights loaded cleanly.")

model = model.to(device)
model.eval()

# Shared normalisation stats (used for all models)
stats_name = f"radar_norm_res{out_res}_log{int(LOG_SCALE_INPUT)}.npz"
stats_candidates = [CKPT_DIR / stats_name, NOTEBOOK_DIR / stats_name]
stats_path = next((p for p in stats_candidates if p.exists()), None)
if stats_path is not None:
    with np.load(stats_path) as stats:
        CHANNEL_MEAN = stats["mean"].astype(np.float32)
        CHANNEL_STD = stats["std"].astype(np.float32)
    print(f"Loaded channel stats from {stats_path}")
else:
    CHANNEL_MEAN = CHANNEL_STD = None
    print("Warning: normalisation stats not found; falling back to per-frame normalisation.")

# --- optionally load ResNet18 checkpoint ---
resnet_model = None
if CKPT_DIR_RESNET is not None:
    resnet_ckpt_path = CKPT_DIR_RESNET / "resnet_pose_best.pt"
    if resnet_ckpt_path.exists():
        print(f"Loading ResNet checkpoint from {resnet_ckpt_path}")
        resnet_ckpt = torch.load(resnet_ckpt_path, map_location=device)
        resnet_state = resnet_ckpt["model"]
        resnet_out_res = int(resnet_ckpt.get("out_res", out_res))
        if resnet_out_res != out_res:
            print(f"Warning: ResNet out_res={resnet_out_res} differs from baseline out_res={out_res}; using baseline out_res for inputs.")
        resnet_model = ResNetPoseNet(num_kp=num_kp, in_ch=2, out_res=resnet_out_res, arch="resnet18", pretrained=False, train_backbone=True)
        try:
            missing_r, unexpected_r = resnet_model.load_state_dict(resnet_state, strict=False)
        except RuntimeError as err:
            raise RuntimeError("Failed to load ResNet checkpoint weights into the architecture") from err
        if missing_r or unexpected_r:
            print("ResNet state dict discrepancies detected during load:")
            if missing_r:
                print(" - Missing keys:", missing_r)
            if unexpected_r:
                print(" - Unexpected keys:", unexpected_r)
        else:
            print("ResNet checkpoint weights loaded cleanly.")
        resnet_model = resnet_model.to(device)
        resnet_model.eval()
    else:
        print(f"Warning: ResNet checkpoint not found at {resnet_ckpt_path}; ResNet predictions disabled.")
else:
    print("Warning: ResNet checkpoint directory not found; ResNet predictions disabled.")


Detected legacy checkpoint; instantiating LegacyRadarPoseNet
Checkpoint weights loaded cleanly.
Loaded channel stats from m:\MIAMI\projects\codex_new\internship projects\Pose Detucting AI using Radardata\checkpoint\radar_norm_res128_log1.npz


C:\Users\ASUS\AppData\Local\Temp\ipykernel_23432\2457437146.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)


In [6]:
def prepare_sample(frame_id: str) -> dict:
    radar = load_radar_frame(DATA_DIR, frame_id, out_res, log_scale=LOG_SCALE_INPUT)
    norm = normalise_channels(radar, CHANNEL_MEAN, CHANNEL_STD)
    norm = np.nan_to_num(norm, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    gt_xy, vis = load_pose_sample(DATA_DIR, frame_id)
    if gt_xy is not None:
        scale_x = out_res / float(CAMERA_WIDTH)
        scale_y = out_res / float(CAMERA_HEIGHT)
        gt_xy = gt_xy * np.array([scale_x, scale_y], dtype=np.float32)
        gt_xy = np.clip(gt_xy, 0.0, out_res - 1)

    mask_img = load_mask_frame(DATA_DIR, frame_id, out_res)

    return {
        "frame_id": frame_id,
        "radar": radar,
        "input": norm,
        "gt_xy": gt_xy,
        "vis": vis,
        "mask": mask_img,
    }


def predict_frame(frame_id: str) -> dict:
    sample = prepare_sample(frame_id)
    tensor = torch.from_numpy(sample["input"]).unsqueeze(0).to(device)
    with torch.no_grad():
        _, coords_cnn, conf_cnn = model(tensor)
        sample["pred_xy_cnn"] = coords_cnn[0].cpu().numpy()
        sample["conf_cnn"] = conf_cnn[0].cpu().numpy()
        # Backwards-compatible keys (baseline CNN)
        sample["pred_xy"] = sample["pred_xy_cnn"]
        sample["conf"] = sample["conf_cnn"]

        if resnet_model is not None:
            _, coords_resnet, conf_resnet = resnet_model(tensor)
            sample["pred_xy_resnet"] = coords_resnet[0].cpu().numpy()
            sample["conf_resnet"] = conf_resnet[0].cpu().numpy()

    return sample


def plot_prediction(sample: dict) -> None:
    gt_xy = sample.get("gt_xy")
    vis = sample.get("vis")
    mask_img = sample.get("mask")
    pred_xy_cnn = sample.get("pred_xy_cnn")
    conf_cnn = sample.get("conf_cnn")
    pred_xy_resnet = sample.get("pred_xy_resnet")
    conf_resnet = sample.get("conf_resnet")

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    ax_gt = axes[0, 0]
    ax_cnn = axes[0, 1]
    ax_res = axes[1, 0]
    ax_all = axes[1, 1]

    def _prepare_bg(ax):
        if mask_img is not None:
            bg = mask_img
            if bg.shape[0] != out_res or bg.shape[1] != out_res:
                bg = cv2.resize(bg, (out_res, out_res), interpolation=cv2.INTER_NEAREST)
        else:
            bg = np.zeros((out_res, out_res), dtype=np.float32)
        ax.imshow(bg, cmap="gray")
        ax.set_xlim(0, out_res - 1)
        ax.set_ylim(out_res - 1, 0)
        ax.set_aspect("equal")
        ax.axis("off")

    def draw_pose(ax, coords: np.ndarray, mask: np.ndarray, color: str, label: str):
        for i, j in SKELETON_EDGES:
            if mask[i] and mask[j]:
                ax.plot(
                    [coords[i, 0], coords[j, 0]],
                    [coords[i, 1], coords[j, 1]],
                    color=color,
                    linewidth=2,
                    alpha=0.9,
                )
        if coords.ndim == 2:
            ax.scatter(coords[mask, 0], coords[mask, 1], c=color, s=30, label=label)

    # Panel 1: Ground truth only
    _prepare_bg(ax_gt)
    if gt_xy is not None:
        if vis is not None:
            mask_gt = vis > 0.5
        else:
            mask_gt = np.ones(gt_xy.shape[0], dtype=bool)
        draw_pose(ax_gt, gt_xy, mask_gt, "lime", "Ground Truth")
    ax_gt.set_title("Ground Truth")

    # Panel 2: CNN only
    _prepare_bg(ax_cnn)
    if pred_xy_cnn is not None:
        if vis is not None:
            mask_cnn = vis > 0.1
        else:
            mask_cnn = np.ones(pred_xy_cnn.shape[0], dtype=bool)
        draw_pose(ax_cnn, pred_xy_cnn, mask_cnn, "cyan", "Baseline CNN")
    ax_cnn.set_title("Baseline CNN")

    # Panel 3: ResNet18 only
    _prepare_bg(ax_res)
    if pred_xy_resnet is not None:
        if vis is not None:
            mask_res = vis > 0.1
        else:
            mask_res = np.ones(pred_xy_resnet.shape[0], dtype=bool)
        draw_pose(ax_res, pred_xy_resnet, mask_res, "magenta", "ResNet18")
    ax_res.set_title("ResNet18")

    # Panel 4: All three overlaid
    _prepare_bg(ax_all)
    if gt_xy is not None:
        if vis is not None:
            mask_gt = vis > 0.5
        else:
            mask_gt = np.ones(gt_xy.shape[0], dtype=bool)
        draw_pose(ax_all, gt_xy, mask_gt, "lime", "GT")
    if pred_xy_cnn is not None:
        if vis is not None:
            mask_cnn = vis > 0.1
        else:
            mask_cnn = np.ones(pred_xy_cnn.shape[0], dtype=bool)
        draw_pose(ax_all, pred_xy_cnn, mask_cnn, "cyan", "CNN")
    if pred_xy_resnet is not None:
        if vis is not None:
            mask_res = vis > 0.1
        else:
            mask_res = np.ones(pred_xy_resnet.shape[0], dtype=bool)
        draw_pose(ax_all, pred_xy_resnet, mask_res, "magenta", "ResNet18")

    # Confidence text on combined panel
    y_text = out_res + 2
    if conf_cnn is not None:
        ax_all.text(0, y_text, f"CNN mean conf: {float(conf_cnn.mean()):.2f}")
        y_text += 6
    if conf_resnet is not None:
        ax_all.text(0, y_text, f"ResNet mean conf: {float(conf_resnet.mean()):.2f}")

    ax_all.legend(loc="upper right")
    fig.suptitle(sample["frame_id"])
    plt.tight_layout()
    plt.show()


In [7]:

# Interactive controls for iterating through frames
if not FRAME_IDS:
    raise RuntimeError("No frames available - cannot run inference.")

CURRENT_INDEX = max(0, min(FRAME_INDEX, len(FRAME_IDS) - 1))

output_area = widgets.Output()


def render_frame(idx: int) -> None:
    idx = max(0, min(idx, len(FRAME_IDS) - 1))
    frame_id = FRAME_IDS[idx]
    with output_area:
        clear_output(wait=True)
        print(f"Using frame index {idx} / {len(FRAME_IDS) - 1}: {frame_id}")
        sample = predict_frame(frame_id)
        plot_prediction(sample)
        start = max(0, idx - 3)
        end = min(len(FRAME_IDS), idx + 4)
        print("Nearby frame ids:")
        for j in range(start, end):
            marker = "<--" if j == idx else "   "
            print(f"{j:04d}: {FRAME_IDS[j]} {marker}")
    index_widget.value = idx


def on_prev(_):
    render_frame(index_widget.value - 1)


def on_next(_):
    render_frame(index_widget.value + 1)


def on_switch(_):
    if not FRAME_IDS:
        return
    render_frame((index_widget.value + 1) % len(FRAME_IDS))


def on_index_change(change):
    if change['name'] == 'value' and change['new'] != change['old']:
        render_frame(change['new'])


index_widget = widgets.BoundedIntText(
    value=CURRENT_INDEX,
    min=0,
    max=len(FRAME_IDS) - 1,
    description='Frame idx:',
    continuous_update=False,
)
prev_button = widgets.Button(description='Previous', icon='arrow-left', button_style='')
next_button = widgets.Button(description='Next', icon='arrow-right', button_style='')
switch_button = widgets.Button(description='Switch Frame', icon='refresh', button_style='')

prev_button.on_click(on_prev)
next_button.on_click(on_next)
switch_button.on_click(on_switch)
index_widget.observe(on_index_change, names='value')

controls = widgets.HBox([prev_button, next_button, switch_button, index_widget])
display(controls, output_area)

render_frame(CURRENT_INDEX)


Output()